In [1]:
!pip install --upgrade pip setuptools wheel
!pip install "numpy<2" "pandas<2.2" "scipy<1.11" "scikit-learn<1.4" "lightgbm==3.4.1"
!pip install pycaret

import pandas as pd
from pycaret.classification import *
from sklearn.model_selection import train_test_split

SEED = 128

  Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl.metadata (53 kB)
  Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl.metadata (11 kB)
ERROR: Ignored the following versions that require a different python version: 1.14.0 Requires-Python >=3.10; 1.14.0rc1 Requires-Python >=3.10; 1.14.0rc2 Requires-Python >=3.10; 1.14.1 Requires-Python >=3.10; 1.15.0 Requires-Python >=3.10; 1.15.0rc1 Requires-Python >=3.10; 1.15.0rc2 Requires-Python >=3.10; 1.15.1 Requires-Python >=3.10; 1.15.2 Requires-Python >=3.10; 1.15.3 Requires-Python >=3.10; 1.16.0 Requires-Python >=3.11; 1.16.0rc1 Requires-Python >=3.11; 1.16.0rc2 Requires-Python >=3.11; 1.16.1 Requires-Python >=3.11; 1.16.2 Requires-Python >=3.11; 1.16.3 Requires-Python >=3.11; 1.7.0 Requires-Python >=3.10; 1.7.0rc1 Requires-Python >=3.10; 1.7.1 Requires-Python >=3.10; 1.7.2 Requires-Python >=3.10
ERROR: Could not find a version that satisfies the requirement lightgbm==3.4.1 (from versions: 2.0.2, 2.0.3, 2.0.4, 2.0.5, 2.

In [2]:
df = pd.read_csv("../Dataset1_UK_Housing/5_price_paid_records_final.csv")
df_sample = df.sample(50000, random_state=SEED) # Take a sample from dataset to train faster

In [3]:
train_df, val_df = train_test_split(
    df_sample, 
    test_size=0.3, 
    random_state=SEED, 
    stratify=df_sample["property_type"]
)

clf = setup(
    data=train_df,
    target="property_type",
    session_id=SEED,
    fold=2,
    verbose=True
)

best_model = compare_models(sort="F1", n_select=1, turbo=True)

tuned_model = tune_model(
    best_model, 
    optimize="F1", 
    fold=5, 
    n_iter=20
)

final_model = finalize_model(tuned_model)

predictions = predict_model(final_model, data=val_df)

from sklearn.metrics import accuracy_score, f1_score

def eval_preds(pred):
    y_true = pred["property_type"]
    y_pred = pred["prediction_label"]
    return accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro")

eval_final = eval_preds(predictions)
print(eval_final)

save_model(final_model, "../Model_UKHousing/best_property_type_model")

,Description,Value
0,Session id,128
1,Target,property_type
2,Target type,Multiclass
3,Target mapping,"D: 0, F: 1, O: 2, S: 3, T: 4"
4,Original data shape,"(35000, 8)"
5,Transformed data shape,"(35000, 8)"
6,Transformed train set shape,"(24500, 8)"
7,Transformed test set shape,"(10500, 8)"
8,Numeric features,1
9,Categorical features,6


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.6104,0.8260,0.6104,0.6005,0.6045,0.4783,0.4788,3.0500
gbc,Gradient Boosting Classifier,0.6040,0.0000,0.6040,0.5912,0.5960,0.4708,0.4716,1.8600
rf,Random Forest Classifier,0.5801,0.8003,0.5801,0.5746,0.5767,0.4374,0.4376,0.2050
et,Extra Trees Classifier,0.5608,0.7709,0.5608,0.5570,0.5587,0.4115,0.4116,0.1850
dt,Decision Tree Classifier,0.5304,0.6809,0.5304,0.5313,0.5307,0.3699,0.3699,0.1150
lr,Logistic Regression,0.5287,0.0000,0.5287,0.5114,0.4897,0.3603,0.3801,1.2650
lda,Linear Discriminant Analysis,0.5084,0.0000,0.5084,0.4880,0.4828,0.3439,0.3511,0.1000
knn,K Neighbors Classifier,0.4562,0.6963,0.4562,0.4611,0.4550,0.2740,0.2756,0.6250
ridge,Ridge Classifier,0.5088,0.0000,0.5088,0.4823,0.4539,0.3390,0.3592,0.5350
ada,Ada Boost Classifier,0.4734,0.0000,0.4734,0.4480,0.3915,0.3202,0.3840,0.6100


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.6263,0.8378,0.6263,0.6154,0.6195,0.5000,0.5007
1,0.6271,0.8366,0.6271,0.6169,0.6207,0.5007,0.5015
2,0.6292,0.8379,0.6292,0.6179,0.6222,0.5034,0.5042
3,0.6347,0.8404,0.6347,0.6239,0.6276,0.5106,0.5117
4,0.6231,0.8345,0.6231,0.6112,0.6156,0.4960,0.4968
Mean,0.6281,0.8374,0.6281,0.6171,0.6211,0.5021,0.5030
Std,0.0038,0.0019,0.0038,0.0041,0.0039,0.0049,0.0049


Fitting 5 folds for each of 20 candidates, totalling 100 fits


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.6359,0.8450,0.6359,0.6255,0.6293,0.5125,0.5134


[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] bagging_fraction is set=1.0, subsample=1.0 will be ignored. Current value: bagging_fraction=1.0
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] bagging_fraction is set=1.0, subsample=1.0 will be ignored. Current value: bagging_fraction=1.0
(0.6359333333333334, 0.6316141933532862)
Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('label_encoding',
                  TransformerWrapperWithInverse(exclude=None, include=None,
                                                transformer=LabelEncoder())),
                 ('numerical_imputer',
                  TransformerWrapper(exclude=None, include=['price'],
                                     transformer=SimpleImputer(add_indicator=False,
                                                               copy=True,
                                                               fill_value=None,
                                                               keep_empty_features=False,
                                                               missing_values=nan,
                                                               strategy='mean...
                                 boosting_type='gbdt', class_weight=None,
                                 colsample_bytree=1.0, feature_fraction=0.6,
                  